# Exercício 1: Silver de Matrículas

Notebook de exploração da tabela Delta gerada pelo pipeline `src/lakehouse_bronze_matriculas.py`.

Tabela: `data/silver/silver_matriculas` (1 linha por `aluno_id`, refletindo o estado mais recente e mais confiável conhecido, com merge idempotente).

In [1]:
import sys

# permite importar os módulos de src/ quando o notebook roda a partir de notebooks/
sys.path.insert(0, "../src")

from lakehouse_bronze_matriculas import create_spark_session

spark = create_spark_session(app_name="notebook_exercicio_1")
silver_path = "../data/silver/silver_matriculas"

:: loading settings :: url = jar:file:/Users/jpdagostin/Desktop/personal/lakehouse_education/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/jpdagostin/.ivy2.5.2/cache
The jars for the packages stored in: /Users/jpdagostin/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fe747c47-262d-45d9-bc02-7d679468730d;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core

	found io.delta#delta-kernel-api;4.3.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.3.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xeria

26/08/08 20:56:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Schema da tabela

In [2]:
df_silver = spark.read.format("delta").load(silver_path)
df_silver.printSchema()

root
 |-- ultimo_evento_id: string (nullable = true)
 |-- aluno_id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- turma: string (nullable = true)
 |-- status: string (nullable = true)
 |-- data_evento: timestamp (nullable = true)
 |-- fonte_ingestao: string (nullable = true)
 |-- dt_processamento_silver: timestamp (nullable = true)



## Dados consolidados (1 linha por aluno_id)

In [3]:
df_silver.orderBy("aluno_id").show(truncate=False)

26/08/08 20:56:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------+--------+-----------+-----+-------+-------------------+--------------+--------------------------+
|ultimo_evento_id|aluno_id|nome       |turma|status |data_evento        |fonte_ingestao|dt_processamento_silver   |
+----------------+--------+-----------+-----+-------+-------------------+--------------+--------------------------+
|ev003           |1001    |Maria Silva|7B   |ativo  |2026-01-10 09:15:00|cdc           |2026-08-06 21:53:31.909605|
|ev006           |1002    |João Souza |8A   |inativo|2026-01-20 11:30:00|cdc           |2026-08-06 21:53:31.909605|
|ev009           |1003    |Ana Costa  |9A   |ativo  |2026-01-07 07:50:00|cdc           |2026-08-06 21:53:31.909605|
+----------------+--------+-----------+-----+-------+-------------------+--------------+--------------------------+



## Histórico de versões (time travel via Delta log)

Cada execução do pipeline gera uma nova versão na tabela (overwrite na primeira execução, merge/upsert nas seguintes).

In [4]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, silver_path).history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+--------------------------------------+
|version|timestamp              |operation|operationParameters                   |
+-------+-----------------------+---------+--------------------------------------+
|0      |2026-08-06 21:53:34.504|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-----------------------+---------+--------------------------------------+

